# Variable Substitution

SLayer lets callers inject `{variable}` placeholders at query time. This is what
makes a dataset or query **reusable** across values (a region, a date window, a
threshold) without string-concatenating SQL in client code.

Substitution works across SLayer's **two expression layers**:

- **Mode B — the DSL** (query `filter`): you filter on dimension/metric names.
- **Mode A — raw SQL** (a stored dataset's `sql` and `filters`, and each field's dialect
  `expression` entry and `filter`): the value binds as a literal of that SQL, *before*
  aggregation. This is the primitive for parameterizing hand-written dataset SQL
  (for example, importing a Cube `FILTER_PARAMS` cube).

This notebook demonstrates every surface end-to-end against the Jaffle Shop
demo. See also: [Variables in dataset SQL](../../concepts/datasets.md#variables-in-dataset-sql)
· [Queries](../../concepts/queries.md#variables) · [References](../../concepts/references.md).

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.models import SlayerDataset

engine, storage, datasets = ensure_jaffle_shop()


def save(dataset: dict) -> str:
    """Save ``dataset`` as a stored dataset; queries name it."""
    engine.run_sync(engine.save_dataset(SlayerDataset.model_validate(dataset)))
    return dataset["name"]

Database created at demo/jaffle_shop.duckdb


## Mode B — query-level filters (the DSL side)

A query `filter` references dimension/metric names, and `{var}` placeholders are
filled from the query's `variables`. One template, many values — no string
building on the caller side.

In [2]:
# One filter template, reused with different values.
for region in ["Brooklyn", "Philadelphia"]:
    result = engine.execute_sync(query={
        "population": "orders",
        "metrics": ["count(orders.*)", "sum(orders.order_total)"],
        "dimensions": ["stores.name"],
        "filter": ["stores.name = '{region}'"],
        "variables": {"region": region},
    })
    row = result.data[0]
    print(f"{region:<14}{row['orders._count']:>10,} orders   ${row['orders.order_total_sum']:>14,.2f}")

Brooklyn         254,821 orders   $  2,745,216.47
Philadelphia     189,367 orders   $  2,228,908.20


### Expression surfaces

Metric expressions, computed dimensions, `order` expressions and `date_range` bounds take
placeholders too; an unnamed metric is named from its template, so the result key
(`orders.order_total_sum_rate_100`) stays the same whatever the value.

In [3]:
# A fee at a given rate: one expression template, many rates.
for rate in [1, 2.5]:
    result = engine.execute_sync(query={
        "population": "orders",
        "metrics": ["sum(orders.order_total) * {rate} / 100"],
        "variables": {"rate": rate},
    })
    print(f"rate {rate:>4}%  ->  {result.data[0]}")

rate    1%  ->  {'orders.order_total_sum_rate_100': 73865.0945}
rate  2.5%  ->  {'orders.order_total_sum_rate_100': 184662.73625}


## Mode A — raw SQL surfaces

Everything below binds the value as a literal of the SQL *inside* a stored dataset,
so it takes effect **before** aggregation — something a post-aggregation query filter
can't do. There are four raw-SQL surfaces; we cover each in turn.

### `SlayerDataset.filters` — a dataset-level `WHERE`

Raw SQL lives only in a stored dataset, so we save one first. The `{floor}` value
lands in the always-applied `WHERE`.

In [4]:
orders_over_floor = save({
    "name": "orders_over_floor",
    "sql_table": "orders",
    "data_source": "jaffle_shop",
    "filters": ["order_total >= {floor}"],  # Mode A: raw SQL, filled from {floor}
    "fields": [
        {"name": "id", "datatype": "String", "primary_key": True},
        {"name": "order_total", "datatype": "Float"},
    ],
})

for floor in [0, 50]:
    result = engine.execute_sync(query={
        "population": orders_over_floor,
        "metrics": ["count(orders_over_floor.*)"],
        "variables": {"floor": floor},
    })
    print(f"order_total >= {floor:>3}: {result.data[0]['orders_over_floor._count']:>10,} orders")

# The value was bound into the SQL before it ran:
where = [line.strip() for line in result.sql.splitlines() if ">=" in line][0]
print("\nGenerated WHERE:", where)

order_total >=   0:    651,446 orders
order_total >=  50:     16,169 orders

Generated WHERE: orders_over_floor.order_total >= 50


### `SlayerDataset.sql` — a full raw-SQL dataset body

An `sql`-mode dataset's body can reference `{var}` anywhere. Below `{floor}` appears
twice — once in the `WHERE` and once as a projected scalar — both filled from the
same variable.

In [5]:
floored_orders = save({
    "name": "floored_orders",
    "data_source": "jaffle_shop",
    "sql": (
        "SELECT id, order_total, {floor} AS floor_used "
        "FROM orders WHERE order_total >= {floor}"
    ),
    "fields": [
        {"name": "id", "datatype": "String", "primary_key": True},
        {"name": "order_total", "datatype": "Float"},
        {"name": "floor_used", "datatype": "Float"},
    ],
})

result = engine.execute_sync(query={
    "population": floored_orders,
    "metrics": ["count(floored_orders.*)"],
    "dimensions": ["floored_orders.floor_used"],
    "variables": {"floor": 60},
})
print(result.data[0])

{'floored_orders.floor_used': 60, 'floored_orders._count': 9115}


### A field's dialect `expression` — a derived field expression

A field's entry for its datasource's dialect is raw SQL too. Here a stored dataset's derived field scales
`order_total` by a runtime `{mult}`.

In [6]:
orders_scaled = save({
    "name": "orders_scaled",
    "sql_table": "orders",
    "data_source": "jaffle_shop",
    "fields": [
        {"name": "id", "datatype": "String", "primary_key": True},
        {"name": "order_total", "datatype": "Float"},
        {"name": "scaled_total", "datatype": "Float",
         "expression": {"dialects": [{"dialect": "DUCKDB", "expression": "order_total * {mult}"}]}},
    ],
})

result = engine.execute_sync(query={
    "population": orders_scaled,
    "metrics": [
        {"metric": "sum(orders_scaled.order_total)", "alias": "raw"},
        {"metric": "sum(orders_scaled.scaled_total)", "alias": "scaled"},
    ],
    "variables": {"mult": 2},
})
row = result.data[0]
print(f"raw sum      = ${row['raw']:,.2f}")
print(f"scaled (x2)  = ${row['scaled']:,.2f}")

raw sum      = $7,386,509.45
scaled (x2)  = $14,773,018.90


### `DatasetField.filter` — a CASE-WHEN at aggregation time

`DatasetField.filter` decides which rows contribute to *that field's* aggregate — the
natural home for windowed sums (e.g. this-year vs last-year) with scalar bounds.
Only rows passing `order_total >= {floor}` feed `big_total`.

In [7]:
orders_big = save({
    "name": "orders_big",
    "sql_table": "orders",
    "data_source": "jaffle_shop",
    "fields": [
        {"name": "id", "datatype": "String", "primary_key": True},
        {"name": "order_total", "datatype": "Float"},
        {"name": "big_total", "expression": "order_total", "filter": "order_total >= {floor}", "datatype": "Float"},
    ],
})

result = engine.execute_sync(query={
    "population": orders_big,
    "metrics": [
        {"metric": "sum(orders_big.order_total)", "alias": "all_orders"},
        {"metric": "sum(orders_big.big_total)", "alias": "big_orders"},
    ],
    "variables": {"floor": 50},
})
row = result.data[0]
print(f"SUM over all orders            = ${row['all_orders']:,.2f}")
print(f"SUM over orders >= $50 (CASE)  = ${row['big_orders']:,.2f}")

SUM over all orders            = $7,386,509.45
SUM over orders >= $50 (CASE)  = $1,132,131.67


## Variable precedence

The same variable can be set in several places. Highest priority wins:

**runtime `variables=` kwarg > stage `variables` > outer query `variables` (when a query-backed dataset is the `population`) > dataset `query_variables`**

`query_variables` on a dataset are the lowest-priority *defaults*.

In [8]:
p_orders = save({
    "name": "p_orders",
    "data_source": "jaffle_shop",
    "query_variables": {"floor": 100},  # lowest-priority default
    "sql": "SELECT id, order_total FROM orders WHERE order_total >= {floor}",
    "fields": [
        {"name": "id", "datatype": "String", "primary_key": True},
        {"name": "order_total", "datatype": "Float"},
    ],
})


def count(query_var=None, runtime=None):
    q = {"population": p_orders, "metrics": ["count(p_orders.*)"]}
    if query_var is not None:
        q["variables"] = {"floor": query_var}
    kwargs = {"variables": {"floor": runtime}} if runtime is not None else {}
    return engine.execute_sync(query=q, **kwargs).data[0]["p_orders._count"]


print(f"dataset default   (floor=100) : {count():>8,}")
print(f"query variable  (floor=50)  : {count(query_var=50):>8,}")
print(f"runtime kwarg   (floor=200) : {count(query_var=50, runtime=200):>8,}   <- overrides the query's 50")

dataset default   (floor=100) :      110
query variable  (floor=50)  :   16,169
runtime kwarg   (floor=200) :        0   <- overrides the query's 50


## Missing variables & escaping

A `{var}` with no value (once any variable is in play) **raises** — a
parameterized dataset is meant to fail loudly, not silently match nothing. And
every value binds as a literal: an embedded quote stays inside the literal you
wrote, and a value that would change the SQL's structure is rejected.

In [9]:
# Missing variable -> clear error.
try:
    engine.execute_sync(query={
        "population": orders_over_floor,
        "metrics": ["count(orders_over_floor.*)"],
        "variables": {"unrelated": 1},  # 'floor' is missing
    })
except ValueError as e:
    print("Missing variable raises:")
    print(" ", e)

# A hostile value in a bare slot is rejected before any SQL runs.
try:
    engine.execute_sync(query={
        "population": orders_over_floor,
        "metrics": ["count(orders_over_floor.*)"],
        "variables": {"floor": "0 OR 1=1"},
    })
except ValueError as e:
    print("\nA non-literal value raises:")
    print(" ", e)

# Embedded quote is escaped (O'Brien -> O''Brien) so the SQL stays valid.
result = engine.execute_sync(query={
    "population": "orders",
    "metrics": ["count(orders.*)"],
    "dimensions": ["stores.name"],
    "filter": ["stores.name = '{name}'"],
    "variables": {"name": "O'Brien"},
})
literal = [line.strip() for line in result.sql.splitlines() if "Brien" in line][0]
print("\nValue O'Brien is escaped in the generated SQL:")
print(" ", literal)

Missing variable raises:
  Undefined variable 'floor' in Dataset 'orders_over_floor' filters[0]: 'order_total >= {floor}'. Available variables: ['unrelated']



A non-literal value raises:
  variables supply literal values: variable 'floor' holds the text '0 OR 1=1' in a number or boolean slot; put the placeholder inside quotes ('{floor}') to use it as a string.

Value O'Brien is escaped in the generated SQL:
  stores.name = 'O''Brien'


## Optional blocks — a filter that vanishes when its value is absent

Every surface above **requires** its variables. A Cube `FILTER_PARAMS` pushdown,
though, is *optional*: it filters when the caller supplies a value and becomes a
no-op when they don't. SLayer expresses that with an **optional block**
`{? ... ?}` on a Mode-A surface — it renders (parenthesised) when every `{var}`
inside is supplied, and collapses to the neutral `(1=1)` otherwise. A **list**
value renders an injection-safe `IN`-list (write the parens; *string* elements
are auto-quoted, numbers stay unquoted). Open the `WHERE` with `1=1` so the
collapse leaves valid SQL. This is exactly how `slayer import-cube` represents an
optional Cube `FILTER_PARAMS` pushdown.

In [10]:
orders_by_store = save({
    "name": "orders_by_store",
    "data_source": "jaffle_shop",
    "sql": (
        "SELECT o.id, o.order_total, s.name AS store_name "
        "FROM orders o LEFT JOIN stores s ON o.store_id = s.id "
        "WHERE 1=1 AND {? s.name IN ({stores}) ?}"
    ),
    "fields": [
        {"name": "id", "datatype": "String", "primary_key": True},
        {"name": "order_total", "datatype": "Float"},
        {"name": "store_name", "datatype": "String"},
    ],
})

# Omitted -> the block collapses to (1=1): every order counts.
allc = engine.execute_sync(query={"population": orders_by_store, "metrics": ["count(orders_by_store.*)"]})

# Supplied (a list) -> renders `s.name IN ('Brooklyn', 'Philadelphia')`.
some = engine.execute_sync(query={
    "population": orders_by_store,
    "metrics": ["count(orders_by_store.*)"],
    "variables": {"stores": ["Brooklyn", "Philadelphia"]},
})

print(f"no filter (block collapses to (1=1)) : {allc.data[0]['orders_by_store._count']:>10,}")
print(f"stores IN [Brooklyn, Philadelphia]   : {some.data[0]['orders_by_store._count']:>10,}")
clause = [ln.strip() for ln in some.sql.splitlines() if "IN (" in ln][0]
print("\nGenerated (list -> IN-list):", clause)

no filter (block collapses to (1=1)) :    651,446
stores IN [Brooklyn, Philadelphia]   :    444,188

Generated (list -> IN-list): s.name IN ('Brooklyn', 'Philadelphia')


## Scope & summary

| Surface | Layer | Example |
|---|---|---|
| query `filter` | Mode B (DSL) | `"stores.name = '{region}'"` |
| metrics, computed dimensions, `order`, `date_range` | Mode B (DSL) | `"sum(orders.order_total) * {rate} / 100"` |
| `SlayerDataset.filters` | Mode A (raw SQL) | `"order_total >= {floor}"` |
| `SlayerDataset.sql` | Mode A (raw SQL) | `"... WHERE order_total >= {floor}"` |
| `DatasetField.sql` | Mode A (raw SQL) | `"order_total * {mult}"` |
| `DatasetField.filter` | Mode A (raw SQL) | `"order_total >= {floor}"` |
| optional block | Mode A (raw SQL) | `"... AND {? s.name IN ({stores}) ?}"` |

**Contract:** raise-on-missing once any variable is in play — for *required*
placeholders; a `{var}` **inside a `{? ... ?}` block** instead collapses the
block to `(1=1)` when absent. A fully variable-free execution leaves braces as
literals (so raw brace literals like a Postgres array `'{1,2,3}'` survive); use
`{{` / `}}` for literal braces in a dataset that *does* use variables. A **list**
value renders an injection-safe `IN`-list body (*string* elements auto-quoted,
numbers unquoted) — together the primitives for representing Cube `FILTER_PARAMS`
pushdowns.

**Scope:** substitution applies to every dataset a query reads, each with its own
defaults under the query's variables. Every value binds as a literal, spelled in the datasource dialect's own
escaping, so no value can change the structure of the SQL it binds into.

In [11]:
# Remove the demo datasets again, so the shared Jaffle Shop store keeps its own.
for name in [orders_over_floor, floored_orders, orders_scaled, orders_big, p_orders, orders_by_store]:
    engine.run_sync(storage.delete_dataset(name, data_source="jaffle_shop"))